In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("dataset/indian_movies.csv")

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.isnull().sum()

,0
id,0
title,0
industry,0
genre,3599
overview,1286


In [ ]:
df = df.dropna()

In [ ]:
df.isnull().sum()

,0
id,0
title,0
industry,0
genre,0
overview,0


In [ ]:
df.columns

Index(['id', 'title', 'industry', 'genre', 'overview'], dtype='object')

In [ ]:
columns = ['title', 'industry', 'genre', 'overview']

In [ ]:
df = df[df['title'].astype(str).apply(lambda x: x.isascii())]

In [ ]:
df.shape

(18312, 5)

In [ ]:
df["features"] = df["title"]+" "+ df["industry"]+ " " + df["genre"] + " " + df["overview"]

In [ ]:
df["features"]

,features
0,"Parthiban Kanavu Kollywood Drama, History 7th ..."
1,"Kalathur Kannamma Kollywood Romance, Drama A l..."
2,Padikkadha Medhai Kollywood Drama Chandrasekar...
3,Paadhai Theriyudhu Paar Kollywood Drama Paadha...
5,"Adutha Veettu Penn Kollywood Romance, Comedy M..."
...,...
22959,The Wheel of the sky Bollywood Drama At a city...
22960,"SAMA Bollywood Drama, Family, Music A Boy's Jo..."
22961,Mast Mahila Mandali Bollywood Documentary Ten ...
22963,Stars at Noon Bollywood Documentary A family i...


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df["features"].tolist(),
    show_progress_bar=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/573 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

print(similarity_matrix.shape)

(18312, 18312)


In [ ]:
similarity_matrix

array([[1.0000001 , 0.4483469 , 0.4497264 , ..., 0.1593969 , 0.18267271,
        0.19728541],
       [0.4483469 , 0.9999999 , 0.49614713, ..., 0.2702515 , 0.26952213,
        0.33524916],
       [0.4497264 , 0.49614713, 1.        , ..., 0.2552654 , 0.24308616,
        0.34671965],
       ...,
       [0.1593969 , 0.2702515 , 0.2552654 , ..., 1.0000002 , 0.41465303,
        0.49143058],
       [0.18267271, 0.26952213, 0.24308616, ..., 0.41465303, 0.99999976,
        0.35378683],
       [0.19728541, 0.33524916, 0.34671965, ..., 0.49143058, 0.35378683,
        0.99999994]], dtype=float32)

In [ ]:
similarity_matrix[2]

array([0.4497264 , 0.49614713, 1.        , ..., 0.2552654 , 0.24308616,
       0.34671965], dtype=float32)

In [ ]:
df = df.reset_index(drop=True)
df_for_indices = df.drop_duplicates(subset=['title'], keep='first')
movie_indices = pd.Series(
    df_for_indices.index,
    index=df_for_indices['title']
)

In [ ]:
def recommend_movies(movie_title, top_n=10):

    if movie_title not in movie_indices:
        return f"Movie '{movie_title}' not found."

    idx = movie_indices[movie_title]

    similarity_scores = list(
        enumerate(similarity_matrix[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    movie_ids = [i[0] for i in similarity_scores]

    recommendations = df.iloc[movie_ids][
        ['title', 'genre', 'industry']
    ]

    return recommendations

In [ ]:
recommend_movies("Youth", top_n=5)

,title,genre,industry
4944,Love Today,"Romance, Drama",Kollywood
17917,Sweety Naughty Crazy,"Romance, Comedy, Drama",Kollywood
7325,Madurai Veeran,"Romance, Drama, Action",Kollywood
8166,Paiyaa,"Action, Adventure, Romance",Kollywood
8525,180,"Drama, Romance",Kollywood


In [ ]:
recommend_movies("With Love", top_n=5)

,title,genre,industry
8553,Eththan,Romance,Kollywood
4944,Love Today,"Romance, Drama",Kollywood
17886,29,"Romance, Comedy, Drama",Kollywood
15122,Theera Kaadhal,"Family, Comedy, Romance",Kollywood
8201,Kaadhal Solla Vandhen,"Action, Drama",Kollywood


In [ ]:
import pickle

with open("movie_recommender.pkl", "wb") as f:
    pickle.dump({
        "df": df,
        "embeddings": embeddings,
        "movie_indices": movie_indices
    }, f)

In [ ]:
import numpy as np
df.to_csv("movies_processed.csv", index=False)
np.save("embeddings.npy", embeddings)